# 정답 노트북


In [ ]:
# CSV 자동 준비 (Colab · JupyterLite · 로컬 공통)
# 파일이 없으면 GitHub raw에서 받아 cwd에 저장 → pd.read_csv("파일명") 그대로 동작
import sys
from pathlib import Path

def ensure_csv(filename, repo_path):
    candidates = [
        Path(filename),
        Path(repo_path),
        Path("files") / repo_path,
        Path("/files") / repo_path,
        Path(repo_path).name,
    ]
    for c in candidates:
        try:
            if c.is_file():
                if not Path(filename).exists():
                    Path(filename).write_bytes(c.read_bytes())
                return filename
        except Exception:
            pass
    url = f"https://raw.githubusercontent.com/aaronlee09-max/informatics/main/{repo_path}"
    try:
        if sys.platform == "emscripten":
            from pyodide.http import open_url
            Path(filename).write_text(open_url(url).read(), encoding="utf-8")
        else:
            import urllib.request
            urllib.request.urlretrieve(url, filename)
        print("CSV 준비:", filename)
        return filename
    except Exception as e:
        print("CSV 준비 실패 → URL 직접 사용:", url, e)
        return url

ensure_csv('penguins_size.csv', 'ai/ch2-2/3/penguins_size.csv')
ensure_csv('food.csv', 'ai/ch2-2/3/food.csv')
ensure_csv('airline_passenger_satisfaction.csv', 'ai/ch2-2/3/airline_passenger_satisfaction.csv')

import pandas as pd
df = pd.read_csv('penguins_size.csv')
df.dropna(inplace=True)
X = df[['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm']]
y = df['species']


In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X = scaler.fit_transform(X)


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
lr = LogisticRegression(max_iter=1000)
knn = KNeighborsClassifier(n_neighbors=5)
lr.fit(X_train, y_train)
knn.fit(X_train, y_train)


In [ ]:
print('로지스틱 회귀:', lr.score(X_test, y_test))
print('kNN:', knn.score(X_test, y_test))


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
pred = lr.predict(X_test)
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=lr.classes_, yticklabels=lr.classes_)
plt.xlabel('예측')
plt.ylabel('실제')
plt.show()
